# Explore here

### Carga del conjunto de datos.

In [65]:
# Your code here
import pandas as pd
import numpy as np

# Cargamos el dataset directamente desde el link de la academia
url = "https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv"
df = pd.read_csv(url)

# Vemos las primeras filas para confirmar que cargó bien
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


### Preprocesamiento de datos. 

In [66]:
# Primero cambiaremos los valores ? por NaN para poder trabajar con ellos
df = df.replace('?', pd.NA)# pd.NA es la forma de representar valores faltantes en pandas

# Ahora podemos ver cuántos valores faltantes hay en cada columna
print(df.isnull().sum())


age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64


In [67]:
# Nuestra variable clave es occupation, eliminaremos todos los registros que no tengan occupation
df = df.dropna(subset=['occupation'])

# Para workclass y native-country, reemplazaremos los valores faltantes por la moda de cada columna
for columna in ['workclass', 'native.country']:
    moda = df[columna].mode()[0]
    df[columna] = df[columna].fillna(moda)


#Como borramos algunas filas, el índice del dataframe ya no es consecutivo, por lo que lo reiniciaremos
df = df.reset_index(drop=True)

# Ahora podemos ver cuántos valores faltantes hay en cada columna
print(df.isnull().sum())


age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64


In [68]:
# Eliminamos la coluumnas que no nos afectan para el análisis, como fnlwgt o education, ya que education.num es la versión numérica de education
df = df.drop(columns=['fnlwgt', 'education'])

In [69]:
# ahora convertiremos la variable income a binaria, 1 si es >50K y 0 si es <=50K
df['income'] = df['income'].str.strip()
df['income_binaria'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)
df = df.drop(columns=['income'])  # ya no necesitamos el texto, tenemos la versión binaria

print(df['income_binaria'].value_counts())
# Copia  para poder leer las recomendaciones después
df_legible = df.copy()

income_binaria
0    23068
1     7650
Name: count, dtype: int64


In [70]:
# Ahora la transformación de variables categóricas 
# Cojeremos las columnas categóricas y las convertiremos a variables dummy (0 o 1) para poder trabajar con ellas en modelos de machine learning
columnas_categoricas = ['workclass', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country']

# con pd.get_dummies podemos convertir las columnas categóricas a variables dummy es decir 1 o 0, 
# y le pasamos el parámetro drop_first=True para que no nos genere una columna de más, 
# ya que si tenemos n categorías, solo necesitamos n-1 columnas para representarlas
df = pd.get_dummies(df, columns=columnas_categoricas)

df.head()

,age,education.num,capital.gain,capital.loss,hours.per.week,income_binaria,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
0,82,9,0,4356,18,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
1,54,4,0,3900,40,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
2,41,10,0,3900,40,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
3,34,9,0,3770,45,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
4,38,6,0,3770,40,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False


In [71]:
# normaliza las variables numéricas.
from sklearn.preprocessing import StandardScaler
# cogeremos las columnas numéricas y las normalizaremos a 0 y 1 para que no tengan un peso mayor en el modelo de machine learning
columnas_numericas = ['age', 'education.num', 'hours.per.week', 'capital.gain', 'capital.loss']

# Con  StandardScaler y fit_transform podemos normalizar las columnas numéricas a 0 y 1
scaler = StandardScaler()
df[columnas_numericas] = scaler.fit_transform(df[columnas_numericas])

df.head()

,age,education.num,capital.gain,capital.loss,hours.per.week,income_binaria,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
0,3.320351,-0.441111,-0.147516,10.519126,-1.914806,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
1,1.185882,-2.392386,-0.147516,9.395006,-0.079207,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
2,0.194878,-0.050856,-0.147516,9.395006,-0.079207,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
3,-0.338739,-0.441111,-0.147516,9.074533,0.337974,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
4,-0.033815,-1.611876,-0.147516,9.074533,-0.079207,0,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False


### Define el problema de recomendación

¿Qué se quiere recomendar? He decidido que voy a recoemndar si conviene cambiar de trabajo es decir ocupación
¿Cuál será el "usuario" en este caso? Cada fila o registro del dataset representa un usuario
¿Qué variables definen el perfil de un usuario?Las variables que utilizas para medir la similitud, es decir, todas las de columnas_vector.

### Construye el sistema de recomendación. (filtrado basado en contenido)

In [72]:
# Primero crearemos una lista con las columnas que son ocupaciones, para poder separarlas de las demás columnas
columnas_ocupacion = [c for c in df.columns if c.startswith('occupation_')]
# Luego necesitamos una lista con las columnas que no son ocupaciones, para poder usarlas en el cálculo de similitud
columnas_vector = [c for c in df.columns if c not in columnas_ocupacion and c != 'income_binaria']

# Las similitud  sirve para medir qué tan similares son dos personas, en este caso, dos registros del dataset.
# comparamos cada registro con todos los demás registros, y obtenemos un valor entre 0 y 1, donde 1 significa que son idénticos y 0 significa que no tienen nada en común.
print(f"Variables usadas para medir similitud: {len(columnas_vector)}")

Variables usadas para medir similitud: 73


In [73]:
# Buscamos la columna income binaria y buscamos las filas que itengan 1, es decir, las personas que ganan más de 50K
# Y nos quedamos con las filas que tengan income_binaria = 1, es decir, las personas que ganan más de 50K
exitosos = df[df['income_binaria'] == 1].reset_index(drop=True)
#Hacemos lo mismo con el dataframe legible, para poder ver los resultados de manera más clara
exitosos_legible = df_legible[df_legible['income_binaria'] == 1].reset_index(drop=True)

print(f"Perfiles >50K disponibles para comparar: {len(exitosos)}")
# Nos saldra que hay 7650 perfiles que ganan más de 50K, y que podemos usar para comparar con el resto de los perfiles y ver qué tan similares

Perfiles >50K disponibles para comparar: 7650


In [80]:
from sklearn.metrics.pairwise import cosine_similarity

def recomendar_ocupacion(vector_usuario, ocupacion_actual, top_n=15):
    # Calculamos la similitud entre el usuario y todos los perfiles exitosos (>50K)
    similitudes = cosine_similarity(vector_usuario, exitosos[columnas_vector])[0]
    # Las ordenamos de mayor a menor y nos quedamos con los índices de los top_n perfiles más similares
    indices_top = np.argsort(similitudes)[::-1][:top_n]
    
    # Obtenemos las ocupaciones de esos perfiles similares
    ocupaciones_similares = exitosos_legible.loc[indices_top, 'occupation']
    # Contamos cuántas veces aparece cada ocupación y nos quedamos con la más común
    recomendacion = ocupaciones_similares.value_counts().idxmax()
    
    # Si la recomendación es la misma que la ocupación actual, indicamos que no se recomienda cambiar
    if recomendacion == ocupacion_actual:
        return f"Tu ocupación actual ('{ocupacion_actual}') ya coincide con el perfil típico de quienes ganan >50K. No se recomienda cambiar."
    else:
        return f"Se recomienda considerar cambiar de '{ocupacion_actual}' a '{recomendacion}', la ocupación más común entre perfiles similares al tuyo que ganan >50K."

In [ ]:
# Aqui pondriamos el índice del usuario que queremos analizar, por ejemplo, el usuario 22
usuario = 22  # cambia el índice para probar con otro usuario

#Sacariamos la ocupación actual del usuario para poder compararla con la recomendación
ocupacion_actual = df_legible.loc[usuario, 'occupation']
# Pasamos el vector del usuario al modelo de recomendación, que es el registro del usuario con todas las columnas excepto la ocupación y el income_binaria
vector_usuario = df[columnas_vector].iloc[[usuario]]
# Y finalmente llamamos a la función de recomendación y mostramos el resultado
resultado = recomendar_ocupacion(vector_usuario, ocupacion_actual)
print(resultado)

Tu ocupación actual ('Craft-repair') ya coincide con el perfil típico de quienes ganan >50K. No se recomienda cambiar.
